<a href="https://colab.research.google.com/github/humbeaniket2006-max/-brand-knowledge-builder/blob/main/Brand_Knowledge_Builder_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Brand Knowledge Base Auto-Builder
> **PineOS** — Automated brand onboarding tool

Paste any D2C brand URL → auto-extracts product catalog, FAQs, policies, and tone of voice → outputs a structured JSON knowledge pack ready for any AI pipeline.

**Stack:** Python · BeautifulSoup · Gemini 2.0 Flash Lite  
**Built by:** Aniket Humbe

---
### How to use
1. Run **Cell 1** to install dependencies
2. Run **Cell 2** to paste your Gemini API key
3. Run **Cell 3** — enter a brand URL and hit Extract
4. Run **Cell 4** to download the JSON output




In [ ]:
# @title ⚙️ Cell 1 — Install Dependencies (run once)
!pip install -q requests beautifulsoup4 google-genai
print("✅ Dependencies installed.")

✅ Dependencies installed.


In [ ]:
# @title 🔑 Cell 2 — Enter Your Gemini API Key
from google.colab import userdata
import getpass

# Try Colab Secrets first (recommended), else prompt
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    if not GEMINI_API_KEY:
        raise KeyError
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    GEMINI_API_KEY = getpass.getpass("Paste your Gemini API key (hidden): ")
    print("✅ API key set.")

print("💡 Tip: Store it permanently via Colab → Secrets (🔑 icon in left sidebar) → key name: GEMINI_API_KEY")

✅ API key loaded from Colab Secrets.
💡 Tip: Store it permanently via Colab → Secrets (🔑 icon in left sidebar) → key name: GEMINI_API_KEY


In [ ]:
# @title 🔧 Cell 3 — Core Functions (run once to load)
import json, re, time, sys
from urllib.parse import urljoin, urlparse
import requests
from bs4 import BeautifulSoup
from google import genai

# ── Config ────────────────────────────────────────────────────────────────────
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}
PRIORITY_SLUGS = [
    "", "/",
    "/about", "/about-us", "/our-story",
    "/faq", "/faqs", "/help", "/support",
    "/shipping", "/shipping-policy",
    "/returns", "/return-policy", "/refund-policy",
    "/privacy-policy",
    "/contact", "/contact-us",
]
MAX_PAGES           = 8
REQUEST_DELAY       = 0.8
TIMEOUT             = 10
GEMINI_MODEL        = "gemini-2.0-flash-lite"
GEMINI_RPM_DELAY    = 4.5
GEMINI_MAX_RETRIES  = 4
GEMINI_BACKOFF_BASE = 10
GEMINI_CHARS_CHUNK  = 4000

# ── Scraping helpers ──────────────────────────────────────────────────────────
def normalize_url(url):
    if not url.startswith(("http://", "https://")):
        url = "https://" + url
    return url.rstrip("/")

def fetch_page(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        resp.raise_for_status()
        return BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        print(f"  [skip] {url} → {e}")
        return None

def extract_text(soup):
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav"]):
        tag.decompose()
    text = soup.get_text(separator=" ", strip=True)
    return re.sub(r"\s{2,}", " ", text)[:6000]

def discover_pages(base_url, soup):
    found = set()
    for a in soup.find_all("a", href=True):
        href = a["href"].lower()
        for slug in PRIORITY_SLUGS[2:]:
            if slug.strip("/") in href:
                full = urljoin(base_url, a["href"])
                if urlparse(full).netloc == urlparse(base_url).netloc:
                    found.add(full.split("?")[0])
    return list(found)[:MAX_PAGES - 1]

# ── Rule-based fallback ───────────────────────────────────────────────────────
def rule_based_extract(pages_text, base_url):
    domain   = urlparse(base_url).netloc.replace("www.", "")
    brand    = domain.split(".")[0].title()
    all_text = " ".join(pages_text.values())
    tone_keywords = {
        "playful":     ["fun", "playful", "yay", "awesome", "love"],
        "premium":     ["luxury", "premium", "exclusive", "crafted", "artisan"],
        "clinical":    ["clinically", "dermatologist", "tested", "proven", "formula"],
        "sustainable": ["sustainable", "eco", "green", "planet", "organic"],
        "empowering":  ["empower", "confident", "you deserve", "for you"],
    }
    detected_tones = [t for t, kws in tone_keywords.items()
                      if any(kw.lower() in all_text.lower() for kw in kws)]
    shipping_hints, return_hints = [], []
    for p in [r"free shipping.*?\d+", r"deliver.*?(\d+)[\s-]*day",
              r"ships.*?within.*?\d+", r"express delivery"]:
        m = re.search(p, all_text, re.IGNORECASE)
        if m: shipping_hints.append(m.group(0)[:120])
    for p in [r"(\d+)[\s-]*day.*?return", r"return.*?(\d+)[\s-]*day",
              r"no[\s-]?return", r"exchange.*?(\d+)[\s-]*day"]:
        m = re.search(p, all_text, re.IGNORECASE)
        if m: return_hints.append(m.group(0)[:120])
    product_hints = []
    for url, text in pages_text.items():
        if any(x in url for x in ["/product", "/shop", "/collection", "/"]):
            candidates = re.findall(r"\b[A-Z][a-z]+(?: [A-Z][a-z]+)*\b", text)
            freq = {}
            for c in candidates: freq[c] = freq.get(c, 0) + 1
            product_hints += [k for k, v in freq.items() if v >= 2 and len(k) > 4]
    return {
        "brand_name": brand, "website": base_url,
        "tone_of_voice": detected_tones or ["neutral"],
        "products_and_categories": list(set(product_hints))[:20],
        "shipping_policy": shipping_hints[0] if shipping_hints else "Not detected",
        "return_policy":   return_hints[0]   if return_hints   else "Not detected",
        "faqs": [], "key_usp": [],
        "extraction_method": "rule-based (no AI)",
    }

# ── Gemini helpers ────────────────────────────────────────────────────────────
def _call_gemini_with_retry(client, prompt):
    for attempt in range(GEMINI_MAX_RETRIES):
        try:
            time.sleep(GEMINI_RPM_DELAY)
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
            return response.text.strip()
        except Exception as e:
            err = str(e).lower()
            if any(x in err for x in ["429", "quota", "rate", "resource_exhausted"]):
                wait = GEMINI_BACKOFF_BASE * (2 ** attempt)
                print(f"  ⏳ Rate limit hit (attempt {attempt+1}/{GEMINI_MAX_RETRIES}). Waiting {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"Rate limit persisted after {GEMINI_MAX_RETRIES} retries.")

def _parse_json_response(raw):
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    return json.loads(raw.strip())

CHUNK_PROMPT = """
You are a brand intelligence analyst. Extract ONLY what is clearly present in
this single page of a brand website. Return valid JSON with these keys
(use null if not found on this page):

{{"products_and_categories": [...], "key_usp": [...], "shipping_policy": "...",
  "return_policy": "...", "faqs": [{{"question": "...", "answer": "..."}}], "tone_signals": [...]}}

Return ONLY JSON — no markdown, no commentary.
[PAGE: {page_url}]
{content}
"""

MERGE_PROMPT = """
You are a brand intelligence analyst. Merge and deduplicate these partial extractions
from a brand website into one clean knowledge pack.

Return ONLY valid JSON:
{{"brand_name": "...", "website": "...", "tone_of_voice": [...],
  "products_and_categories": [...], "key_usp": [...],
  "shipping_policy": "...", "return_policy": "...",
  "faqs": [{{"question": "...", "answer": "..."}}],
  "brand_persona_summary": "..."}}

Website: {url}
Partial extractions:
{partials}
"""

def gemini_extract(pages_text, base_url, api_key):
    client   = genai.Client(api_key=api_key)
    pages    = list(pages_text.items())[:MAX_PAGES]
    partials = []
    print(f"  🤖 {GEMINI_MODEL} — {len(pages)} pages, {GEMINI_RPM_DELAY}s delay between calls")
    for i, (page_url, text) in enumerate(pages):
        print(f"  → Page {i+1}/{len(pages)}: {page_url}")
        prompt = CHUNK_PROMPT.format(page_url=page_url, content=text[:GEMINI_CHARS_CHUNK])
        try:
            raw     = _call_gemini_with_retry(client, prompt)
            partial = _parse_json_response(raw)
            partials.append(partial)
        except json.JSONDecodeError:
            print(f"    [warn] Non-JSON for {page_url} — skipping.")
        except RuntimeError as e:
            print(f"    [warn] {e} — stopping AI extraction.")
            break
        except Exception as e:
            print(f"    [warn] {e} — skipping page.")
    if not partials:
        print("  ⚠️  All Gemini calls failed. Falling back to rule-based.")
        return rule_based_extract(pages_text, base_url)
    print(f"  🔀 Merging {len(partials)} partials...")
    partials_str = json.dumps(partials, indent=2, ensure_ascii=False)[:8000]
    merge_prompt = MERGE_PROMPT.format(url=base_url, partials=partials_str)
    try:
        raw    = _call_gemini_with_retry(client, merge_prompt)
        result = _parse_json_response(raw)
        result["extraction_method"] = GEMINI_MODEL
        return result
    except Exception as e:
        print(f"  [warn] Merge failed: {e}. Rule-based fallback.")
        return rule_based_extract(pages_text, base_url)

# ── Main orchestrator ─────────────────────────────────────────────────────────
def extract_brand(url, api_key=None, use_ai=True):
    base_url  = normalize_url(url)
    pages_text = {}
    print(f"\n🔍 Crawling: {base_url}")
    home_soup = fetch_page(base_url)
    if home_soup is None:
        raise ValueError(f"Could not reach {base_url}")
    pages_text[base_url] = extract_text(home_soup)
    discovered = discover_pages(base_url, home_soup)
    slug_urls  = [base_url + s for s in PRIORITY_SLUGS[2:]]
    candidates = list(dict.fromkeys(slug_urls + discovered))
    print(f"  Found {len(candidates)} candidate pages...")
    for page_url in candidates[:MAX_PAGES - 1]:
        if page_url in pages_text: continue
        time.sleep(REQUEST_DELAY)
        soup = fetch_page(page_url)
        if soup:
            pages_text[page_url] = extract_text(soup)
            print(f"  ✓ {page_url}")
    print(f"\n  Scraped {len(pages_text)} pages total.")
    if use_ai and api_key:
        print("  🤖 Running Gemini extraction...")
        return gemini_extract(pages_text, base_url, api_key)
    else:
        print("  🔧 Running rule-based extraction...")
        return rule_based_extract(pages_text, base_url)

print("✅ All functions loaded. Proceed to Cell 4.")

✅ All functions loaded. Proceed to Cell 4.


In [ ]:
# @title 🚀 Cell 4 — Run Extraction

# @markdown ### Settings
BRAND_URL = "mamaearth.in" # @param {type:"string"}
USE_AI    = True           # @param {type:"boolean"}

# Run
brand_pack = extract_brand(
    url     = BRAND_URL,
    api_key = GEMINI_API_KEY if USE_AI else None,
    use_ai  = USE_AI
)

# ── Pretty summary ────────────────────────────────────────────────────────────
sep = "─" * 55
print(f"\n{sep}")
print(f"  ✅ Brand Knowledge Pack — {brand_pack.get('brand_name', 'Unknown')}")
print(sep)
print(f"  Tone of voice   : {', '.join(brand_pack.get('tone_of_voice') or [])}")
print(f"  Products found  : {len(brand_pack.get('products_and_categories') or [])} categories")
print(f"  FAQs extracted  : {len(brand_pack.get('faqs') or [])} entries")
print(f"  Shipping policy : {(brand_pack.get('shipping_policy') or 'N/A')[:80]}")
print(f"  Return policy   : {(brand_pack.get('return_policy') or 'N/A')[:80]}")
print(f"  Method          : {brand_pack.get('extraction_method')}")
print(sep)

# Store for next cell
LAST_RESULT = brand_pack


🔍 Crawling: https://mamaearth.in
  Found 17 candidate pages...
  ✓ https://mamaearth.in/about
  ✓ https://mamaearth.in/about-us
  ✓ https://mamaearth.in/our-story
  ✓ https://mamaearth.in/faq
  ✓ https://mamaearth.in/faqs
  ✓ https://mamaearth.in/help
  ✓ https://mamaearth.in/support

  Scraped 8 pages total.
  🤖 Running Gemini extraction...
  🤖 gemini-2.0-flash-lite — 8 pages, 4.5s delay between calls
  → Page 1/8: https://mamaearth.in
  ⏳ Rate limit hit (attempt 1/4). Waiting 10s...
  ⏳ Rate limit hit (attempt 2/4). Waiting 20s...
  ⏳ Rate limit hit (attempt 3/4). Waiting 40s...
  ⏳ Rate limit hit (attempt 4/4). Waiting 80s...
    [warn] Rate limit persisted after 4 retries. — stopping AI extraction.
  ⚠️  All Gemini calls failed. Falling back to rule-based.

───────────────────────────────────────────────────────
  ✅ Brand Knowledge Pack — Mamaearth
───────────────────────────────────────────────────────
  Tone of voice   : playful, clinical
  Products found  : 20 categories
  FAQs

In [ ]:
# @title 👁️ Cell 5 — View Full JSON Output
import json
print(json.dumps(LAST_RESULT, indent=2, ensure_ascii=False))

{
  "brand_name": "Mamaearth",
  "website": "https://mamaearth.in",
  "tone_of_voice": [
    "playful",
    "clinical"
  ],
  "products_and_categories": [
    "Turmeric",
    "Mamaearth Vitamin",
    "Hair Growth",
    "Hair Fall Control",
    "Natural Lip Balm",
    "Regular",
    "Use Code",
    "Checkout Page Buy Any",
    "Mamaearth Bye Bye Dark Circles Eye",
    "Mamaearth Rice Face Scrub",
    "Tan Removal",
    "Beetroot",
    "Daily Glow Face Serum",
    "Mamaearth Anti Hairfall Spa Kit",
    "Supple Lips",
    "Mamaearth Rice Water Dewy Face Toner",
    "Mamaearth",
    "Free Face Moisturizer With Rice Water",
    "Mamaearth Rosemary Anti",
    "Hair Fall Conditioner"
  ],
  "shipping_policy": "Not detected",
  "return_policy": "Not detected",
  "faqs": [],
  "key_usp": [],
  "extraction_method": "rule-based (no AI)"
}


In [ ]:
# @title 💾 Cell 6 — Save & Download JSON
import json
from google.colab import files
from urllib.parse import urlparse

slug     = urlparse(LAST_RESULT.get("website", "brand")).netloc.replace("www.", "").split(".")[0]
filename = f"{slug}_brand_knowledge.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(LAST_RESULT, f, indent=2, ensure_ascii=False)

print(f"💾 Saved as {filename}")
files.download(filename)
print("✅ Download triggered.")

💾 Saved as mamaearth_brand_knowledge.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download triggered.


In [ ]:
# @title 🔁 Cell 7 (Bonus) — Batch Extract Multiple Brands
# @markdown Add comma-separated URLs below, e.g. `mamaearth.in, nykaa.com, boat-lifestyle.com`

BRAND_URLS_BATCH = "mamaearth.in, bewakoof.com" # @param {type:"string"}
USE_AI_BATCH     = True                          # @param {type:"boolean"}

import json, time
from google.colab import files

urls       = [u.strip() for u in BRAND_URLS_BATCH.split(",") if u.strip()]
all_brands = {}

for i, url in enumerate(urls):
    print(f"\n{'='*55}")
    print(f"[{i+1}/{len(urls)}] Processing: {url}")
    print('='*55)
    try:
        result = extract_brand(
            url     = url,
            api_key = GEMINI_API_KEY if USE_AI_BATCH else None,
            use_ai  = USE_AI_BATCH
        )
        all_brands[url] = result
        print(f"  ✅ Done: {result.get('brand_name', url)}")
    except Exception as e:
        print(f"  ❌ Failed: {e}")
        all_brands[url] = {"error": str(e)}
    # Pause between brands to respect rate limits
    if i < len(urls) - 1:
        print(f"  ⏳ Waiting 15s before next brand (rate limit buffer)...")
        time.sleep(15)

# Save combined output
batch_file = "all_brands_knowledge.json"
with open(batch_file, "w", encoding="utf-8") as f:
    json.dump(all_brands, f, indent=2, ensure_ascii=False)

print(f"\n{'='*55}")
print(f"✅ Batch complete — {len(all_brands)} brands processed")
print(f"💾 Saved to {batch_file}")
files.download(batch_file)


[1/2] Processing: mamaearth.in

🔍 Crawling: https://mamaearth.in
  [skip] https://mamaearth.in → 503 Server Error: Service Unavailable for url: https://mamaearth.com
  ❌ Failed: Could not reach https://mamaearth.in
  ⏳ Waiting 15s before next brand (rate limit buffer)...

[2/2] Processing: bewakoof.com

🔍 Crawling: https://bewakoof.com
  Found 15 candidate pages...
  [skip] https://bewakoof.com/about → 404 Client Error: Not Found for url: https://www.bewakoof.com/404
  [skip] https://bewakoof.com/about-us → 404 Client Error: Not Found for url: https://www.bewakoof.com/404
  [skip] https://bewakoof.com/our-story → 404 Client Error: Not Found for url: https://www.bewakoof.com/404
  ✓ https://bewakoof.com/faq
  [skip] https://bewakoof.com/faqs → 404 Client Error: Not Found for url: https://www.bewakoof.com/404
  [skip] https://bewakoof.com/help → 404 Client Error: Not Found for url: https://www.bewakoof.com/404
  [skip] https://bewakoof.com/support → 404 Client Error: Not Found for url: 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>